# Chemical Space Analysis with TMAP

This notebook generates TMAP (Tree MAP) visualizations for chemical space analysis of DeNovo HsDHODH datasets.

## Notebook Structure:

1. **Imports** - Required libraries (tmap, rdkit, bokeh)
2. **Data Loading** - Reading druglike and highdiv datasets
3. **Fingerprint Functions** - MACCS and ECFP4
4. **create_tmap_data Function** - Calculates TMAP for a dataset
5. **Individual TMAP Calculation** - Processes each dataset separately
6. **Individual Interactive Plots** - Generates HTML visualizations with Bokeh
7. **Combined Visualizations** - Concatenates all datasets, removes duplicates by InChI, and generates visualizations with legend by source dataset

## Processed Datasets:
- **Diversity**: DRUGLIKE, HIGHDIV
- **Descriptors**: MACCS, ECFP4
- **Datasets**: CONCAT, CRAFT, LANA, MAY

**Total**: 10 visualizations (HTML)
- 8 individual
- 2 combined

In [ ]:
import tmap as tm
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from rdkit.Chem import Draw
from bokeh.plotting import figure, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
import base64
from io import BytesIO

In [ ]:
df_druglike = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Datasets/concat_datasets_druglike.csv')
df_highdiv = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Datasets/concat_datasets_highdiv.csv')

In [ ]:
def smiles_to_maccs(smiles):
    """Converts a SMILES into MACCS fingerprint"""
    mol = Chem.MolFromSmiles(smiles)
    return MACCSkeys.GenMACCSKeys(mol) if mol else None

def smiles_to_ECFP4(smiles):
    """Converts a SMILES into ECFP4 fingerprint (Morgan with radius 2)"""
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024) if mol else None

In [ ]:
def create_tmap_data(df, fingerprint_type='maccs'):
    """
    Creates data needed to generate a TMAP
    
    Args:
        df: DataFrame with 'SMILES' column
        fingerprint_type: 'maccs' or 'ecfp4'
    
    Returns:
        layout: TMAP layout data
        df: Updated DataFrame with fingerprints
    """
    # Generate fingerprints
    if fingerprint_type == 'maccs':
        df['fingerprints'] = [smiles_to_maccs(smiles) for smiles in df['SMILES']]
        n_bits = 166  # MACCS has 166 bits
    elif fingerprint_type == 'ecfp4':
        df['fingerprints'] = [smiles_to_ECFP4(smiles) for smiles in df['SMILES']]
        n_bits = 1024  # ECFP4 now has 1024 bits
    else:
        raise ValueError("fingerprint_type must be 'maccs' or 'ecfp4'")
    
    # Remove rows without fingerprints
    df = df.dropna(subset=['fingerprints'])
    
    # Create MinHash
    minhashes = []
    minhash = tm.Minhash(n_bits)
    for fp in df['fingerprints']:
        minhashes.append(minhash.from_binary_array(list(fp)))
    
    # Create LSH Forest
    lf = tm.LSHForest(n_bits, 128)
    lf.batch_add(minhashes)
    lf.index()
    
    # Create k-NN graph
    k = 10
    knng_from = tm.VectorUint()
    knng_to = tm.VectorUint()
    knng_weight = tm.VectorFloat()
    lf.get_knn_graph(knng_from, knng_to, knng_weight, k)
    
    # Create layout
    edges_list = [(knng_from[i], knng_to[i], knng_weight[i]) for i in range(len(knng_from))]
    num_vertices = len(df)
    layout = tm.layout_from_edge_list(num_vertices, edges_list, create_mst=True)
    
    return layout, df

In [ ]:
# Create results directory if it doesn't exist
import os
output_dir = '/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/Analysis/Results/Chemical_Spaces/TMAP'
os.makedirs(output_dir, exist_ok=True)

# Dictionary to store calculated TMAP results
tmap_results = {}

# Generate individual TMAPs for each dataset and descriptor
datasets_info = [
    ('DRUGLIKE', df_druglike),
    ('HIGHDIV', df_highdiv)
]

fingerprint_types = ['MACCS', 'ECFP4']
dataset_names = ['CONCAT', 'CRAFT', 'LANA', 'MAY']

print("="*60)
print("CALCULATING TMAPS")
print("="*60)

for diversity_type, df_full in datasets_info:
    print(f"\n{'='*60}")
    print(f"Processing: {diversity_type}")
    print('='*60)
    
    for dataset_name in dataset_names:
        # Filter specific dataset
        df_subset = df_full[df_full['Dataset'].str.contains(dataset_name, case=False, na=False)].copy()
        
        if len(df_subset) == 0:
            print(f"⚠️  {dataset_name}: No data found")
            continue
            
        print(f"\n{dataset_name}: {len(df_subset)} molecules")
        
        for descriptor in fingerprint_types:
            print(f"  - Calculating TMAP with {descriptor}...", end=' ')
            
            try:
                # Create TMAP
                layout, df_processed = create_tmap_data(df_subset, fingerprint_type=descriptor.lower())
                
                # Store results
                key = f"{diversity_type}_{descriptor}_{dataset_name}"
                tmap_results[key] = {
                    'layout': layout,
                    'df': df_processed,
                    'diversity': diversity_type,
                    'descriptor': descriptor,
                    'dataset': dataset_name
                }
                
                print(f"✓ Complete ({len(df_processed)} molecules)")
                
            except Exception as e:
                print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Calculation complete! {len(tmap_results)} TMAPs calculated.")
print('='*60)

In [ ]:
# Function to convert molecule to base64 image
def mol_to_base64(smiles, size=(150, 150)):
    """Converts SMILES to base64 encoded PNG image"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        img = Draw.MolToImage(mol, size=size)
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return f"data:image/png;base64,{img_str}"
    except:
        return ""

# Generate interactive plots with Bokeh
print("="*60)
print("GENERATING INTERACTIVE PLOTS (HTML)")
print("="*60)

for key, data in tmap_results.items():
    layout = data['layout']
    df_processed = data['df']
    diversity_type = data['diversity']
    descriptor = data['descriptor']
    dataset_name = data['dataset']
    
    print(f"Plotting: {key}...", end=' ')
    
    try:
        # Extract coordinates
        x = [layout[0][i] for i in range(len(df_processed))]
        y = [layout[1][i] for i in range(len(df_processed))]
        
        # Prepare data for Bokeh
        df_plot = df_processed.copy().reset_index(drop=True)
        
        # Fix: Remove fingerprints column which is not serializable
        if 'fingerprints' in df_plot.columns:
            df_plot = df_plot.drop(columns=['fingerprints'])
            
        df_plot['x'] = x
        df_plot['y'] = y
        
        # Generate molecule images
        print("(generating images...", end=' ')
        df_plot['mol_image'] = df_plot['SMILES'].apply(mol_to_base64)
        print("OK)", end=' ')
        
        # Create ColumnDataSource
        source = ColumnDataSource(df_plot)
        
        # Create figure
        p = figure(
            width=900,
            height=700,
            title=f"TMAP - {diversity_type} - {dataset_name} - {descriptor} ({len(df_plot)} molecules)",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            toolbar_location="right"
        )
        
        # Add points
        circles = p.circle(
            'x', 'y',
            size=5,
            alpha=0.6,
            color='steelblue',
            source=source
        )
        
        # Configure HoverTool with molecule image
        hover = HoverTool(
            tooltips="""
            <div style="width:200px;">
                <div>
                    <img src="@mol_image" style="width:150px; height:150px; border:1px solid #ddd; padding:5px;">
                </div>
                <div style="margin-top:5px;">
                    <span style="font-weight:bold;">Dataset:</span> @Dataset<br>
                    <span style="font-weight:bold;">SMILES:</span> @SMILES<br>
                    <span style="font-weight:bold;">MW:</span> @MW<br>
                    <span style="font-weight:bold;">LogP:</span> @LogP
                </div>
            </div>
            """,
            renderers=[circles]
        )
        p.add_tools(hover)
        
        # Configure axis labels
        p.xaxis.axis_label = "Dimension 1"
        p.yaxis.axis_label = "Dimension 2"
        p.title.text_font_size = "14pt"
        
        # Save as HTML
        filename = f"{diversity_type}_{descriptor}_{dataset_name}.html"
        filepath = os.path.join(output_dir, filename)
        output_file(filepath)
        save(p)
        
        print(f"✓ Saved")
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Interactive plots saved in: {output_dir}")
print('='*60)

## Combined Visualization - All Datasets

In this section, we will create visualizations that combine all datasets (CONCAT, CRAFT, LANA, MAY) for each diversity type (DRUGLIKE and HIGHDIV), removing duplicates by InChI and maintaining the source dataset information.

In [ ]:
# Prepare combined data removing duplicates by InChI
def prepare_combined_data(df_full):
    """
    Concatenates all datasets and removes duplicates by InChI,
    keeping source dataset information
    """
    df_combined = df_full.copy()
    
    # Remove molecules without InChI
    df_combined = df_combined.dropna(subset=['InChI'])
    print(f"  Molecules with valid InChI: {len(df_combined)}")
    
    # Remove duplicates by InChI, keeping the first occurrence
    initial_count = len(df_combined)
    df_combined = df_combined.drop_duplicates(subset=['InChI'], keep='first')
    duplicates_removed = initial_count - len(df_combined)
    print(f"  Duplicates removed: {duplicates_removed}")
    print(f"  Unique molecules: {len(df_combined)}")
    
    return df_combined

print("="*60)
print("PREPARING COMBINED DATA")
print("="*60)

# Prepare DRUGLIKE combined
print("\nDRUGLIKE:")
df_druglike_combined = prepare_combined_data(df_druglike)

# Prepare HIGHDIV combined
print("\nHIGHDIV:")
df_highdiv_combined = prepare_combined_data(df_highdiv)

print(f"\n{'='*60}")
print("Preparation complete!")
print('='*60)

In [ ]:
# Calculate TMAPs for combined datasets
print("="*60)
print("CALCULATING COMBINED TMAPS")
print("="*60)

tmap_combined_results = {}

combined_datasets = [
    ('DRUGLIKE', df_druglike_combined),
    ('HIGHDIV', df_highdiv_combined)
]

for diversity_type, df_combined in combined_datasets:
    print(f"\n{'='*60}")
    print(f"Processing: {diversity_type}_COMBINED ({len(df_combined)} molecules)")
    print('='*60)
    
    for descriptor in fingerprint_types:
        print(f"  Calculating TMAP with {descriptor}...", end=' ')
        
        try:
            # Create TMAP
            layout, df_processed = create_tmap_data(df_combined, fingerprint_type=descriptor.lower())
            
            # Store results
            key = f"{diversity_type}_COMBINED_{descriptor}"
            tmap_combined_results[key] = {
                'layout': layout,
                'df': df_processed,
                'diversity': diversity_type,
                'descriptor': descriptor,
                'dataset': 'COMBINED'
            }
            
            print(f"✓ Complete ({len(df_processed)} molecules)")
            
        except Exception as e:
            print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Calculation complete! {len(tmap_combined_results)} combined TMAPs calculated.")
print('='*60)

In [ ]:
# Generate interactive HTML plots for combined datasets with legend by dataset
print("="*60)
print("GENERATING COMBINED INTERACTIVE PLOTS (HTML)")
print("="*60)

from bokeh.models import Legend, LegendItem

# Bokeh colors for each dataset
dataset_colors_bokeh = {
    'CONCAT': '#e74c3c',  # red
    'CRAFT': '#3498db',   # blue
    'LANA': '#2ecc71',    # green
    'MAY': '#f39c12'      # orange
}

for key, data in tmap_combined_results.items():
    layout = data['layout']
    df_processed = data['df']
    diversity_type = data['diversity']
    descriptor = data['descriptor']
    
    print(f"Plotting: {key}...", end=' ')
    
    try:
        # Extract coordinates
        x = [layout[0][i] for i in range(len(df_processed))]
        y = [layout[1][i] for i in range(len(df_processed))]
        
        # Prepare data for Bokeh
        df_plot = df_processed.copy().reset_index(drop=True)
        
        # FIX: Remove fingerprints column which is not serializable
        if 'fingerprints' in df_plot.columns:
            df_plot = df_plot.drop(columns=['fingerprints'])
            
        df_plot['x'] = x
        df_plot['y'] = y
        
        # Generate molecule images
        print("(generating images...", end=' ')
        df_plot['mol_image'] = df_plot['SMILES'].apply(mol_to_base64)
        print("OK)", end=' ')
        
        # Create figure
        p = figure(
            width=1000,
            height=800,
            title=f"Combined TMAP - {diversity_type} - {descriptor} ({len(df_plot)} molecules unique by InChI)",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            toolbar_location="right"
        )
        
        # List to store legend items
        legend_items = []
        
        # Add points by dataset with different colors
        for dataset_name in dataset_names:
            # Filter data for this dataset
            df_subset = df_plot[df_plot['Dataset'].str.contains(dataset_name, case=False, na=False)]
            
            if len(df_subset) > 0:
                # Create ColumnDataSource for this subset
                source = ColumnDataSource(df_subset)
                
                # Add circles
                circles = p.circle(
                    'x', 'y',
                    size=7,
                    alpha=0.7,
                    color=dataset_colors_bokeh[dataset_name],
                    source=source,
                    legend_label=f'{dataset_name} (n={len(df_subset)})'
                )
                
                # Configure HoverTool for this renderer
                hover = HoverTool(
                    tooltips="""
                    <div style="width:200px;">
                        <div>
                            <img src="@mol_image" style="width:150px; height:150px; border:1px solid #ddd; padding:5px;">
                        </div>
                        <div style="margin-top:5px;">
                            <span style="font-weight:bold;">Dataset:</span> @Dataset<br>
                            <span style="font-weight:bold;">SMILES:</span> @SMILES<br>
                            <span style="font-weight:bold;">MW:</span> @MW<br>
                            <span style="font-weight:bold;">LogP:</span> @LogP
                        </div>
                    </div>
                    """,
                    renderers=[circles]
                )
                p.add_tools(hover)
        
        # Configure axis labels
        p.xaxis.axis_label = "Dimension 1"
        p.yaxis.axis_label = "Dimension 2"
        p.title.text_font_size = "14pt"
        
        # Configure legend
        p.legend.location = "top_right"
        p.legend.click_policy = "hide"  # Allows hiding when clicking
        p.legend.background_fill_alpha = 0.8
        
        # Save as HTML
        filename = f"{diversity_type}_COMBINED_{descriptor}.html"
        filepath = os.path.join(output_dir, filename)
        output_file(filepath)
        save(p)
        
        print(f"✓ Saved")
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Combined interactive plots saved in: {output_dir}")
print('='*60)